# SubC Forecast Data — Public ArrayLake Access

Reads SubC forecast data from `ou-subc/subc-forecasts` **without an API token**.

**Required environment (one-time setup):**
```bash
conda create -n subc-arraylake-public python=3.11 -c conda-forge
conda activate subc-arraylake-public
conda install -c conda-forge xarray matplotlib pandas numpy ipykernel cartopy
pip install arraylake
python -m ipykernel install --user --name subc-arraylake-public --display-name "Python (subc-arraylake-public)"
```

**Repository group naming:** `{group}-{model}-forecast`  
**Dimensions:** S (init date), M (member), L (lead day), Y (lat), X (lon), P (pressure, 3D vars only)  
**Variables:** `pr`, `tas`, `rlut`, `ts`, `ua`, `va`, `zg`

## 1. Imports

In [ ]:
%matplotlib inline

from arraylake import Client
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import cartopy
import matplotlib
print(f'cartopy {cartopy.__version__}  matplotlib {matplotlib.__version__}')

## 2. Cartopy smoke test

In [ ]:
proj = ccrs.Robinson(central_longitude=180)
fig, ax = plt.subplots(1, 1, figsize=(8, 4), subplot_kw={'projection': proj})
ax.add_feature(cfeature.COASTLINE)
ax.set_global()
ax.set_title('Cartopy smoke test')
plt.show()
print('Cartopy OK')

## 3. Connect to public repository

In [ ]:
client = Client()
repo = client.get_repo('ou-subc/subc-forecasts')
session = repo.readonly_session('main')
store = session.store
print(f'Connected. Snapshot: {session.snapshot_id}')

## 4. Configuration and helpers

In [ ]:
ALL_GROUPS = {
    'ECCC GEPS8':     'eccc-geps8-forecast',
    'EMC GEFSv12':    'emc-gefsv12_cpc-forecast',
    'ESRL FIMr1p1':   'esrl-fimr1p1-forecast',
    'GMAO GEOS V2p1': 'gmao-geos_v2p1_5daily-forecast',
    'RSMAS CCSM4':    'rsmas-ccsm4-forecast',
}

# Week boundaries in lead days (1-based: day 1 = L index 0)
WEEK_BOUNDS = {
    'Week 1': (0,  7),
    'Week 2': (7,  14),
    'Week 3': (14, 21),
    'Week 4': (21, 28),
}

PROJ      = ccrs.Robinson(central_longitude=180)
TRANSFORM = ccrs.PlateCarree()

KG_TO_MMDAY = 86400.0


def add_map_features(ax):
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle=':')
    ax.set_global()


def weekly_mean(da):
    """Return only weeks that have data."""
    nL = da.sizes['L']
    weeks = {}
    for name, (start, end) in WEEK_BOUNDS.items():
        if start < nL:
            actual_end = min(end, nL)
            weeks[name] = da.isel(L=slice(start, actual_end)).mean(dim='L')
    return weeks


def load_ensemble_mean(group_name, var, init_date=None):
    ds = xr.open_zarr(store, group=group_name, consolidated=False)
    times = pd.DatetimeIndex(ds['S'].values)
    if init_date is None:
        init_date = times[-1]
    da = ds[var].sel(S=init_date).mean(dim='M')
    # Drop duplicate lead times if any
    _, idx = np.unique(da['L'].values, return_index=True)
    da = da.isel(L=idx).compute()
    return da, pd.Timestamp(init_date)


print('Helpers defined.')

## 5. Single model — weekly precipitation maps

In [ ]:
group = 'eccc-geps8-forecast'
var   = 'pr'

pr_mean, init = load_ensemble_mean(group, var)
print(f'Lead times: {pr_mean.sizes["L"]}  range: {pr_mean["L"].values[0]} to {pr_mean["L"].values[-1]}')
print(f'Raw data range: {float(pr_mean.min()):.4e} to {float(pr_mean.max()):.4e}  units: {pr_mean.attrs.get("units", "unknown")}')

if float(pr_mean.max()) < 1.0:
    pr_mean = pr_mean * KG_TO_MMDAY
    print('Converted to mm/day')

wk_maps = weekly_mean(pr_mean)
nwks = len(wk_maps)
ncols = min(nwks, 4)
nrows = int(np.ceil(nwks / ncols))

fig, axes = plt.subplots(
    nrows, ncols, figsize=(6 * ncols, 5 * nrows),
    subplot_kw={'projection': PROJ}
)
axes_flat = np.array(axes).flat

vmin, vmax = 0, 15

for ax, (wk_label, da_wk) in zip(axes_flat, wk_maps.items()):
    add_map_features(ax)
    p = ax.pcolormesh(
        da_wk['X'].values, da_wk['Y'].values, da_wk.values,
        transform=TRANSFORM, cmap='BuPu', vmin=vmin, vmax=vmax,
    )
    ax.set_title(wk_label, fontsize=13)

# Hide any unused axes
for ax in list(axes_flat)[nwks:]:
    ax.set_visible(False)

fig.suptitle(f'ECCC GEPS8 — Ensemble Mean Precipitation\nInit: {init.date()}', fontsize=14)
fig.subplots_adjust(bottom=0.08, top=0.88, hspace=0.05, wspace=0.05)
cbar_ax = fig.add_axes([0.15, 0.02, 0.7, 0.02])
fig.colorbar(p, cax=cbar_ax, orientation='horizontal', label='Precipitation (mm/day)')
plt.show()

## 6. Multi-Model Ensemble (MME) — weekly precipitation maps

In [ ]:
var = 'pr'

# Load ensemble mean for every model, keyed by label
model_data = {}
for label, group_name in ALL_GROUPS.items():
    try:
        da, init = load_ensemble_mean(group_name, var)
        if float(da.max()) < 1.0:
            da = da * KG_TO_MMDAY
        model_data[label] = da
        print(f'  {label}: init {init.date()}  L={da.sizes["L"]}')
    except Exception as e:
        print(f'  {label}: skipped ({e})')

# For each week, average all models that have data covering that week
wk_maps_mme = {}
ref_da = next(iter(model_data.values()))  # borrow coords from any model

for wk_name, (start, end) in WEEK_BOUNDS.items():
    week_arrays = []
    for label, da in model_data.items():
        if da.sizes['L'] > start:           # model has data for this week
            actual_end = min(end, da.sizes['L'])
            wk = da.isel(L=slice(start, actual_end)).mean(dim='L').values
            week_arrays.append(wk)
    if week_arrays:
        mme_wk = np.nanmean(np.stack(week_arrays, axis=0), axis=0)
        wk_maps_mme[wk_name] = ref_da.isel(L=0).copy(data=mme_wk)
        print(f'  {wk_name}: averaged {len(week_arrays)} model(s)')

# Also keep a full-L mme DataArray for the comparison cell
min_L = min(da.sizes['L'] for da in model_data.values())
mme_vals = np.nanmean(
    np.stack([da.isel(L=slice(0, min_L)).values for da in model_data.values()], axis=0),
    axis=0
)
mme = ref_da.isel(L=slice(0, min_L)).copy(data=mme_vals)

# Plot
nwks = len(wk_maps_mme)
fig, axes = plt.subplots(
    1, nwks, figsize=(6 * nwks, 5),
    subplot_kw={'projection': PROJ}
)

vmin, vmax = 0, 15

for ax, (wk_label, da_wk) in zip(np.array(axes).flat, wk_maps_mme.items()):
    add_map_features(ax)
    p = ax.pcolormesh(
        da_wk['X'].values, da_wk['Y'].values, da_wk.values,
        transform=TRANSFORM, cmap='BuPu', vmin=vmin, vmax=vmax,
    )
    ax.set_title(wk_label, fontsize=13)

fig.suptitle('MME — Ensemble Mean Precipitation (all available models)', fontsize=14)
fig.subplots_adjust(bottom=0.12, top=0.88, wspace=0.05)
cbar_ax = fig.add_axes([0.15, 0.04, 0.7, 0.02])
fig.colorbar(p, cax=cbar_ax, orientation='horizontal', label='Precipitation (mm/day)')
plt.show()

## 7. All models side-by-side for a single week

In [ ]:
week_to_plot = 'Week 2'
var = 'pr'

start, end = WEEK_BOUNDS[week_to_plot]

n = len(ALL_GROUPS) + 1  # models + MME
fig, axes = plt.subplots(
    1, n, figsize=(5 * n, 4),
    subplot_kw={'projection': PROJ}
)

vmin, vmax = 0, 15

for ax, (label, group_name) in zip(axes[:-1], ALL_GROUPS.items()):
    try:
        da, init = load_ensemble_mean(group_name, var)
        if float(da.max()) < 1.0:
            da = da * KG_TO_MMDAY
        if da.sizes['L'] > start:
            actual_end = min(end, da.sizes['L'])
            da_wk = da.isel(L=slice(start, actual_end)).mean(dim='L')
            add_map_features(ax)
            ax.pcolormesh(
                da_wk['X'].values, da_wk['Y'].values, da_wk.values,
                transform=TRANSFORM, cmap='BuPu', vmin=vmin, vmax=vmax
            )
            ax.set_title(f'{label}\n{init.date()}', fontsize=10)
        else:
            ax.set_title(f'{label}\n(no data)', fontsize=10)
    except Exception as e:
        ax.set_title(f'{label}\n(skipped)', fontsize=10)
        print(f'{label}: {e}')

# MME panel (uses wk_maps_mme computed in previous cell)
da_wk_mme = wk_maps_mme.get(week_to_plot)
add_map_features(axes[-1])
if da_wk_mme is not None:
    p = axes[-1].pcolormesh(
        da_wk_mme['X'].values, da_wk_mme['Y'].values, da_wk_mme.values,
        transform=TRANSFORM, cmap='BuPu', vmin=vmin, vmax=vmax
    )
axes[-1].set_title('MME', fontsize=10)

fig.suptitle(f'{week_to_plot} — Ensemble Mean Precipitation', fontsize=13)
fig.subplots_adjust(bottom=0.12, top=0.85, wspace=0.05)
cbar_ax = fig.add_axes([0.15, 0.04, 0.7, 0.03])
fig.colorbar(p, cax=cbar_ax, orientation='horizontal', label='Precipitation (mm/day)')
plt.show()